# 🪱 JORINOVA NEXUS — Stool/Urine Parasitology (O&P) detector (fine-tuning)

Fine-tunes a **YOLOv8** detector to find parasite **ova / cysts / larvae** on stool
or urine microscopy and produces `parasitology.pt` for the app's vision service.
Each organism is mapped to its **disease** by the app.

> **Fine-tuning, NOT from scratch** — starts from pretrained weights
> (`yolov8s.pt`, or your blood-domain `malaria.pt`) and adapts them.

**Before you start:** Runtime → **Change runtime type** → **T4 GPU**.

## 1. Check the GPU

In [ ]:
!nvidia-smi -L || echo 'NO GPU — set Runtime > Change runtime type > T4 GPU, then rerun'

## 2. Install training dependencies

In [ ]:
%pip -q install ultralytics roboflow kaggle
import ultralytics, torch
print('ultralytics', ultralytics.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 3. Get the project code + mount Drive

In [ ]:
import os
from getpass import getpass
REPO = 'https://github.com/dujoely1/JORINOVA-NEXUS.git'
tok = getpass('GitHub token (press Enter if the repo is public): ').strip()
url = REPO.replace('https://', f'https://{tok}@') if tok else REPO
os.system(f'rm -rf /content/nexus && git clone --depth 1 {url} /content/nexus')
print('repo cloned:', os.path.exists('/content/nexus/ml/parasitology'))
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/nexus_parasitology', exist_ok=True)
except Exception as e:
    print('Drive not mounted (optional):', e)

## 4. Get the training data — Option A (Roboflow, ready) or Option B (Kaggle Chula)
Option A gives a ready YOLOv8 export (recommended). Option B is the full 11k Chula set.

### ✅ Option A — Roboflow parasite-egg project (ready YOLOv8, needs a FREE key)
Pick a project at https://universe.roboflow.com/search?q=parasite%20egg → Download → YOLOv8.

In [ ]:
# Uncomment and fill in a project, then run:
# from getpass import getpass
# import os
# os.environ['ROBOFLOW_API_KEY'] = getpass('Roboflow Private API Key: ').strip()
# from roboflow import Roboflow
# rf = Roboflow(api_key=os.environ['ROBOFLOW_API_KEY'])
# ds = rf.workspace('WORKSPACE').project('PROJECT').version(1).download('yolov8', location='/content/data/para_roboflow')
# DATA_YAML = os.path.join(ds.location, 'data.yaml'); print('DATA_YAML =', DATA_YAML)

### Option B — Kaggle Chula-ParasiteEgg-11 (11 species, 11k imgs) → COCO→YOLO
Needs `kaggle.json` (kaggle.com → Account → Create New API Token). Uploads it, downloads
the set, then converts COCO boxes to YOLO. Adjust `base` if the unzip layout differs.

In [ ]:
# 1) upload kaggle.json (from your Kaggle account), then download + unzip
# from google.colab import files; files.upload()   # pick kaggle.json
# import os; os.makedirs('/root/.kaggle', exist_ok=True)
# os.replace('kaggle.json', '/root/.kaggle/kaggle.json'); os.chmod('/root/.kaggle/kaggle.json', 0o600)
# !kaggle datasets download -d macharning/chula-parasite-dataset -p /content/data/chula --unzip
#
# 2) COCO -> YOLO conversion (best-effort; inspect the layout if it errors)
import os, glob, json, shutil
from pathlib import Path
from collections import defaultdict
base = '/content/data/chula'
coco_path = None
for j in glob.glob(base + '/**/*.json', recursive=True):
    try:
        d = json.load(open(j))
        if isinstance(d, dict) and {'annotations','images','categories'} <= set(d):
            coco_path = j; break
    except Exception:
        pass
assert coco_path, 'No COCO json found under ' + base + ' — inspect the download layout.'
coco = json.load(open(coco_path))
cat_sorted = sorted(c['id'] for c in coco['categories'])
catname = {c['id']: c['name'] for c in coco['categories']}
names = [str(catname[k]).strip().lower().replace(' ', '_') for k in cat_sorted]
catidx = {k: i for i, k in enumerate(cat_sorted)}
disk = {os.path.basename(p): p for ext in ('*.jpg','*.jpeg','*.png')
        for p in glob.glob(base + '/**/' + ext, recursive=True)}
by_img = defaultdict(list)
for a in coco['annotations']:
    by_img[a['image_id']].append(a)
items = list(coco['images']); OUT = '/content/data/para_yolo'
nval = max(1, len(items)//7)
for k, im in enumerate(items):
    split = 'val' if k < nval else 'train'
    src = disk.get(os.path.basename(im['file_name']))
    if not src: continue
    W, H = im['width'], im['height']
    oi = Path(OUT)/'images'/split; ol = Path(OUT)/'labels'/split
    oi.mkdir(parents=True, exist_ok=True); ol.mkdir(parents=True, exist_ok=True)
    shutil.copy(src, oi/os.path.basename(src))
    lines = []
    for a in by_img.get(im['id'], []):
        x, y, w, h = a['bbox']; cid = catidx[a['category_id']]
        lines.append(f'{cid} {(x+w/2)/W:.6f} {(y+h/2)/H:.6f} {w/W:.6f} {h/H:.6f}')
    (ol/(Path(src).stem + '.txt')).write_text(chr(10).join(lines))
DATA_YAML = os.path.join(OUT, 'data.yaml')
open(DATA_YAML, 'w').write(f'path: {OUT}\ntrain: images/train\nval: images/val\nnc: {len(names)}\nnames: {names}\n')
print('converted ->', DATA_YAML, '| classes:', names)

## 5. Choose the base weights — fine-tuning (never from scratch)

In [ ]:
import os
USE_MALARIA_BASE = False   # True -> transfer from your malaria detector (microscopy features)
malaria_pt = '/content/nexus/backend/models/malaria/malaria.pt'
BASE_WEIGHTS = malaria_pt if (USE_MALARIA_BASE and os.path.exists(malaria_pt)) else 'yolov8s.pt'
print('fine-tuning from:', BASE_WEIGHTS)

## 6. Fine-tune the detector

In [ ]:
EPOCHS = 100
try:
    from ultralytics import YOLO
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'], check=True)
    from ultralytics import YOLO
import os
PROJECT = '/content/runs/detect'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = '/content/drive/MyDrive/nexus_parasitology/runs'; os.makedirs(PROJECT, exist_ok=True)
    print('runs -> Drive:', PROJECT)
except Exception as e:
    print('Drive not mounted, runs stay on /content:', e)
model = YOLO(BASE_WEIGHTS)   # pretrained -> fine-tune; head adapts to the parasite classes
results = model.train(data=DATA_YAML, epochs=EPOCHS, imgsz=640, batch=16,
                      project=PROJECT, name='parasitology', pretrained=True, patience=25, plots=True,
                      hsv_h=0.015, hsv_s=0.6, hsv_v=0.4, degrees=180,
                      fliplr=0.5, flipud=0.5, mosaic=1.0, translate=0.1, scale=0.5)
m = model.val()
print(f'mAP50 = {m.box.map50:.3f}   mAP50-95 = {m.box.map:.3f}')

## 7. Export `parasitology.pt` (app path + Drive backup + download)

In [ ]:
import shutil, glob, os
DRIVE = None
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive/nexus_parasitology'; os.makedirs(DRIVE, exist_ok=True)
except Exception as e:
    print('Drive not mounted (continuing):', e)
best_hits = [c for c in glob.glob('/content/**/weights/best.pt', recursive=True) if os.path.isfile(c)]
last_hits = [c for c in glob.glob('/content/**/weights/last.pt', recursive=True) if os.path.isfile(c)]
backup    = [c for c in glob.glob((DRIVE or '') + '/**/*.pt', recursive=True) if DRIVE and os.path.isfile(c)]
pool = best_hits or last_hits or backup
if not pool:
    raise FileNotFoundError('No trained weights found — run step 6 (training) first.')
best = sorted(pool, key=os.path.getmtime)[-1]
print('using weights:', best, '->', 'best.pt' if best_hits else ('last.pt' if last_hits else 'Drive backup'))
os.makedirs('/content/nexus/backend/models/parasitology', exist_ok=True)
dest = '/content/nexus/backend/models/parasitology/parasitology.pt'; shutil.copy(best, dest); print('saved to app path:', dest)
if DRIVE:
    shutil.copy(best, os.path.join(DRIVE, 'parasitology.pt')); print('backed up to Drive')
try:
    from ultralytics import YOLO
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'], check=True)
    from ultralytics import YOLO
try:
    YOLO(best).export(format='onnx', opset=12)
except Exception as e:
    print('onnx export skipped:', e)
from google.colab import files
files.download(best)

## 8. Put it in the app
1. Move `parasitology.pt` → **`backend/models/parasitology/parasitology.pt`**, commit, push.
2. Auto-loads for `image_type` ∈ {`parasitology`, `stool`, `ova`, `urine_parasite`}.
3. On Render, build with `INSTALL_ML=1` (≥2 GB) to run the detector; else Claude vision reads the field.

Detected organisms → diseases via `backend/ai_services/parasitology_organisms.json`
(hookworm → iron-deficiency anaemia; *Opisthorchis* → cholangiocarcinoma risk; ...).